## making pipeline ingetion

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shreyas jagtap\AppData\Local\Temp\ipykernel_18668\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
d:\shreyas\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(directory_path):
    all_documents=[]
    directory=Path(directory_path)

    pdf_lists=list(directory.glob("**/*.pdf"))

    print(f"The {len(pdf_lists)} no of files will be processed in the directory")

    for pdf_file in pdf_lists:
        print(f"proccessing file {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
               doc.metadata['source_file']=pdf_file.name
               doc.metadata['File_Type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error::{e}")

    print(f"total {len(all_documents)} documents loaded ")
    return all_documents



alll_documents=process_all_pdfs("../data")

The 8 no of files will be processed in the directory
proccessing file amazon sales dashboard.pdf
 Loaded 1 pages
proccessing file ola_booking_dashboard.pdf
 Loaded 1 pages
proccessing file SHREYAS copy sv.pdf
 Loaded 2 pages
proccessing file SHREYAS DT CV.pdf
 Loaded 1 pages
proccessing file shreyas new ds.pdf
 Loaded 1 pages
proccessing file shreyas resume DS.pdf
 Loaded 1 pages
proccessing file shreyas resume.pdf
 Loaded 1 pages
proccessing file certificate deloitt.pdf
 Loaded 1 pages
total 9 documents loaded 


In [3]:
alll_documents

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260622212613', 'source': '..\\data\\dashboard\\amazon sales dashboard.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'amazon sales dashboard.pdf', 'File_Type': 'pdf'}, page_content='Power BI Desktop\ntotal orders\n73.63K\nAverage Purchase Amount\n₹ 643.64\nTotal sale\n47.39M\nAmazon Sales Dashboard\nSum of Amount by ship_state\nMAHARAS…\nKARNATAKA\nUTTAR PR…\nTELANGANA\nTAMIL NADU\nDELHI\nKERALA\nWEST BEN…\nANDHRA P…\nHARYANA\n8.0M\n6.2M\n4.2M\n4.0M\n3.9M\n2.6M\n2.3M\n2.2M\n1.9M\n1.7M\nTotal Qty Sold by Category\nT-sh… Shirt\nBlazzer\nTrousers\nPerfume\nWallet\nSocks\nShoes\n26K 25K\n8K 6K\n1K 1K 0K 0K\nSum of Qty by Status\nShipped 62K (93.5%)\nCancelled 3K (4.3%)\nShipped Cancelled Return\nCount of Cancelled order by ship_state\nMAHARAS…\nKARNATAKA\nUTTAR PRA…\nTELANGANA\nTAMIL\n NADU\nKERALA\nDELHI\nWEST BENG…\nANDHRA P…\nHARYANA\n1.6K\n1.25K\n0.89K 0.86K 0.83K\n0.62K 0.48K 0.47

## now we are splitiing text in the documents

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"Total {len(split_docs)} documents after splitting")
        print(f"content:{split_docs[0].page_content[:200]}")
        print(f"metadata:{split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks=split_documents(alll_documents)


split 9 into 22 chunks
Total 22 documents after splitting
content:Power BI Desktop
total orders
73.63K
Average Purchase Amount
₹ 643.64
Total sale
47.39M
Amazon Sales Dashboard
Sum of Amount by ship_state
MAHARAS…
KARNATAKA
UTTAR PR…
TELANGANA
TAMIL NADU
DELHI
KERAL
metadata:{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260622212613', 'source': '..\\data\\dashboard\\amazon sales dashboard.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'amazon sales dashboard.pdf', 'File_Type': 'pdf'}


In [6]:
type(chunks)

list

## embedding and vector store

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class embedding_manager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._model_load()

    def _model_load(self):
        try:
             print(f"loading the model{self.model_name}")
             self.model=SentenceTransformer(self.model_name)
             print(f"model {self.model_name} loaded successfully. enbedding dimentions:{self.model.get_embedding_dimension()}")    
        except Exception as e:
             print(f"Error occurred while loading the model: {e}")
             raise 
    
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        try:
            print(f"Generating embeddings for {len(texts)} texts")
            embeddings=self.model.encode(texts,show_progress_bar=True)
            print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error occurred while generating embeddings: {e}")
            raise

# initialize the embedding manager
embedding_m=embedding_manager()
embedding_m
embeddings=embedding_m.generate_embeddings([text.page_content for text in chunks])
embeddings

loading the modelall-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2378.14it/s]


model all-MiniLM-L6-v2 loaded successfully. enbedding dimentions:384
Generating embeddings for 22 texts


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.93s/it]

Embeddings generated successfully. Shape: (22, 384)


array([[-0.00650468,  0.03171637, -0.01976004, ..., -0.05948891,
        -0.06660698, -0.02858298],
       [ 0.0725197 ,  0.08046215, -0.00946841, ...,  0.01549768,
        -0.00634959, -0.05103204],
       [-0.01534888,  0.03208304, -0.08001334, ..., -0.0060971 ,
        -0.01716787, -0.01980851],
       ...,
       [ 0.01389693, -0.08314085,  0.00873397, ..., -0.0288518 ,
        -0.03492109, -0.01341984],
       [-0.02889784, -0.05055739, -0.0325853 , ..., -0.04979264,
        -0.07679939, -0.00571307],
       [-0.13718842,  0.00720817, -0.04836875, ..., -0.08725321,
         0.0225098 ,  0.00271524]], shape=(22, 384), dtype=float32)

## vector store

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 110


### JUST CALLING THE FUNCTION AND AADING THE DATA IN VECTOR STORE

In [12]:
vec_store=vectorstore.add_documents(chunks, embeddings)
vec_store

Adding 22 documents to vector store...
Successfully added 22 documents to vector store
Total documents in collection: 132


### creating retrival pipeline

In [48]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
      
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
         
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if True:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_m)

In [49]:
rag_retriever

In [50]:
rag_retriever.retrieve("AMAZON SALES ANALYSIS DASHBOARD")

Retrieving documents for query: 'AMAZON SALES ANALYSIS DASHBOARD'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.80it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a4e0bf81_2',
  'content': 'SHREYAS  JAGTAP  \nshreyasjagtap262004@gmail.com \nLinkedin \nGitHub  \nDhankwadi,pune-43| Ph.no.82650008879  \nObjective \nAspiring Data Analyst with strong skills in Python, SQL, Power BI, and Excel, seeking to leverage data -driven insights and \nanalytical problem-solving to support business decision-making and organizational growth. \nPROJECTS \nAMAZON SALES ANALYSIS DASHBOARD | PYTHON, SQL, POWER BI(ETL,DAX) \n• Cleaned and transformed Amazon sales data using Python (Pandas) by handling missing values, duplicates, and data \ninconsistencies (NumPy,Pnadas,Matplot,Seaborn) \n• Performed SQL analysis to identify sales trends, top-performing categories, state-wise revenue, and order cancellation \npatterns.  \n• Developed an interactive Power BI dashboard to visualize KPIs, sales performance, customer behavior, and business \ninsights. url: https://github.com/shreyasjagtap262004-netizen/Amazon-Sales-Analysis.git \nOLA RIDE BOOKING ANALYSIS DASH

## integreat the vectordb context with llm output


In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("groq_api_key")

In [52]:
llm=ChatGroq(api_key=groq_api_key,model="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## create simple rag function
def rag_sinmple(query,retriever,llm,top_k=3):
    result=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in result])if result else "No relevant documents found."

    prompt=f"""use the following context to answer the question:\n\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [53]:
answer = rag_sinmple(
    "WHO IS SHREYAS JAGTAP",
    rag_retriever,
    llm,
    top_k=3
)
answer

Retrieving documents for query: 'WHO IS SHREYAS JAGTAP'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.45it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


'**Shreyas Jagtap** is a professional who earned a **Certificate of Completion** for a “Data Analytics Job Simulation” program issued by **Forage** on **5\u202fJuly\u202f2026**.  \n\nDuring that month, Shreyas completed practical tasks in:\n\n- **Data analysis**  \n- **Forensic technology**\n\nThe certificate includes verification details:\n\n- **Enrolment Verification Code:** `ASjLYppL7JqvrnHa5`  \n- **User Verification Code:** `6a494a178ab92b5cf693c164`\n\nThe program’s credentials were overseen by **Tina McCreery, Chief Human Resources Officer at Deloitte**.'

### adding some extra features in the rag pipelin

In [54]:
def advance_rag(query,retriver,llm,top_k=3,score_threshold=0.2,return_context=False):
   result=retriver.retrieve(query,top_k=top_k,score_threshold=score_threshold)
   if not result:
    output = {
        "answer": "No relevant documents found.",
        "sources": [],
        "confidence": 0.0
    }

    if return_context:
        output["context"] = ""

    return output
   #prepare context and sources
   context="\n\n".join([doc['content'] for doc in result])
   sources=[{
            'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
            'page':doc['metadata'].get('page','unknown'),
            'similarity_score':doc['similarity_score'],
            'preview':doc['content'][:300]+ "..."} for doc in result]

   confidence = max([doc['similarity_score'] for doc in result])
    
   prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
   response = llm.invoke([prompt.format(context=context, query=query)])

   output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
   if return_context:
    output['context'] = context
   return output

adv_rag=advance_rag("WHO IS SHREYAS JAGTAP",rag_retriever,llm,top_k=3,score_threshold=0.2,return_context=True)
print("Answer:", adv_rag['answer'])
print("Sources:", adv_rag['sources'])
print("Confidence:", adv_rag['confidence'])
print("Context Preview:", adv_rag['context'][:300])


   
   


Retrieving documents for query: 'WHO IS SHREYAS JAGTAP'
Top K: 3, Score threshold: 0.2
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.47it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Shreyas Jagtap is a participant who earned a Data Analytics Job Simulation Certificate of Completion from Forage on July 5, 2026. He completed practical tasks in data analysis and forensic technology, and his enrolment and user verification codes (ASjLYppL7JqvrnHa5 and 6a494a178ab92b5cf693c164) confirm the credential. The certificate is referenced by Tina McCreery, Chief Human Resources Officer at Deloitte.
Sources: [{'source': 'certificate deloitt.pdf', 'page': 0, 'similarity_score': 0.19436222314834595, 'preview': 'shreyas\tjagtap\nData\tAnalytics\tJob\tSimulation\nCertificate\tof\tCompletion\nJuly\t5th,\t2026\nOver\tthe\tperiod\tof\tJuly\t2026,\tshreyas\tjagtap\thas\tcompleted\tpractical\ttasks\tin:\nData\tanalysis\nForensic\ttechnology\nTina\tMcCreery\n\t\nChief\tHuman\nResources\tOfficer,\nDeloitte\nEnrolment\tVerification\tCode\tASjLYppL7JqvrnHa5...'}, {'source': 'certificate deloitt.pdf', 'page': 0, 'similarity_score': 0.19436222314834595, 'preview': 'shreyas\tjagtap\nDa